In [2]:
# ======================
# 全局配置及数据库连接
# ======================

# 解决中文字体问题
# 强制指定非交互式后端（避免某些环境下出现GUI backend错误）
import re
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
import matplotlib
matplotlib.use('TkAgg')  # 强制使用Tkinter图形后端
import matplotlib.pyplot as plt

# 配置中文字体参数
matplotlib.rcParams['font.sans-serif'] = ['KaiTi']  # 设置默认中文字体为楷体
matplotlib.rcParams['font.serif'] = ['KaiTi']       # 设置衬线字体

In [3]:
############################################# 合并数据文件 ##########################################################
"""
合并多个CSV数据文件：
1. 遍历1-8号文件路径
2. 使用pandas读取每个CSV文件
3. 将所有数据拼接为统一DataFrame
"""
dir = r"C:/data-analysis-collection/renting-apartment-beijing"
data_list = []
for i in range(1, 9):
    path = f"{dir}\\bj_danke_{i}.csv"
    data = pd.read_csv(path)
    data_list.append(data)
data = pd.concat(data_list)

In [4]:
############################################### 数据清洗 #############################################################
"""
数据质量处理流程：
1. 删除完全重复的数据行并重置索引
2. 删除包含缺失值的行并重置索引
3. 过滤'户型'字段异常的记录
"""
data.drop_duplicates(inplace=True)
data.reset_index(drop=True, inplace=True)

data.dropna(axis=0, inplace=True)
data.reset_index(drop=True, inplace=True)

data = data[data['户型'] != '户型']

In [5]:
"""
字段格式清洗：
- 移除'地铁'字段中的冗余前缀
- 拆分楼层信息为当前楼层和总楼层
- 使用正则提取地铁线路数量和距离信息
"""
data.loc[:, '地铁'] = data['地铁'].apply(lambda x: x.replace('地铁：', ''))

data.loc[:, '所在楼层'] = data['楼层'].apply(lambda x: int(x.split('/')[0]))
data.loc[:, '总楼层'] = data['楼层'].apply(lambda x: int(x.replace('层', '').split('/')[-1]))
data.loc[:, '地铁数'] = data['地铁'].apply(lambda x: len(re.findall('线', x)))
data.loc[:, '距离地铁距离'] = data['地铁'].apply(
    lambda x: int(re.findall(r'(\d+)米', x)[-1]) if re.findall(r'(\d+)米', x) else -1
)

In [6]:
"""
数据类型转换：
将数值型字段转换为int64类型以优化存储和计算
"""
data['价格'] = data['价格'].astype(np.int64)
data['面积'] = data['面积'].astype(np.int64)
data['距离地铁距离'] = data['距离地铁距离'].astype(np.int64)

In [7]:
################################################## 数据保存 #########################################################
"""
数据持久化配置：
1. 打印数据摘要信息
2. 保存为CSV文件
3. 使用SQLAlchemy引擎存储到SQLite数据库
"""
print(data.info)

engine = create_engine('sqlite:///C:/data-analysis-collection/renting-apartment-beijing/renting-apartment-beijing.db')
data.to_sql('renting-apartment-beijing', con=engine, index=False, if_exists='append')

# 创建SQLite数据库引擎（路径指向北京租房数据库）
engine = create_engine('sqlite:///C:/data-analysis-collection/renting-apartment-beijing/renting-apartment-beijing.db')

<bound method DataFrame.info of         价格   面积        编号    户型      楼层  位置1     位置2       小区  \
0     1330    9   38738-A  3室1卫   9/11层  房山区  良乡大学城西     紫汇家园   
1     5810   39   64752-A  1室1卫   9/12层  东城区    广渠门内      绿景苑   
2     4820   36   65359-A  1室1卫    1/6层  东城区      景泰      定安里   
3     1850    9     185-B  4室2卫   2/27层  朝阳区      管庄  京通苑阳光华苑   
4     9000  116   17830-A  2室1卫   8/16层  朝阳区     十里堡     天天朝阳   
...    ...  ...       ...   ...     ...  ...     ...      ...   
6020  1600   12   79984-B  4室1卫    4/6层  通州区    北运河西     运乔嘉园   
6021  4760   37  150161-A  1室1卫   9/24层  丰台区     刘家窑     远中悦麒   
6022  5920   34   49913-A  1室1卫   6/12层  东城区    广渠门内      绿景苑   
6023  2900   15   62649-C  4室1卫  24/25层  朝阳区      四惠      力源里   
6024  2000   13   69646-A  3室1卫   3/21层  朝阳区    东坝中街      东泽园   

                   地铁  所在楼层  总楼层  地铁数  距离地铁距离  
0     距房山线良乡大学城西站550米     9   11    1     550  
1       距7号线广渠门内站650米     9   12    1     650  
2      距14号线东段景泰站450米     1    6    1     4

In [8]:
# ======================
# 地区房源分析
# ======================
# 查询各行政区的房源总量、小区数和租金均价
sql = """
select 位置1, count(1) as total, count(distinct 小区) as com, sum(价格)/sum(面积) as per
from `renting-apartment-beijing`
group by 位置1
"""
# 绘制行政区房源数量柱状图
data = pd.read_sql(con=engine, sql=sql)
data = data.sort_values(by='total', ascending=False)
plt.bar(data['位置1'], data['total'], label='房源数量')
# 添加数值标签
for x, y in zip(data['位置1'], data['total']):
    plt.text(x, y, y, ha='center', va='bottom', fontsize=11)
plt.legend()
plt.show()

In [9]:
# ======================
# 小区租金排名分析
# ======================
# 查询租金单价最高的前10个小区
sql = """
select 小区, 位置1, count(1) as total, sum(价格)/sum(面积) as per
from `renting-apartment-beijing`
group by 小区, 位置1
order by per desc
limit 10
"""
# 绘制横向条形图展示租金单价排名
data = pd.read_sql(con=engine, sql=sql)
data = data.sort_values(by='per')
plt.barh(data['小区'], data['per'], label='租金(元/平米)', color='g')
# 添加横向数值标签
for x, y in zip(data['小区'], data['per']):
    plt.text(y, x, y, ha='left', va='center', fontsize=11)
plt.legend()
plt.show()

In [10]:
# ======================
# 户型分布分析
# ======================
# 按户型分类统计（截取户型前3个字符）
sql = """
select substr(户型, 0, 3) as 户型, count(1) as total, sum(价格)/sum(面积) as per
from `renting-apartment-beijing`
group by substr(户型, 0, 3)
order by 1
"""
# 绘制户型分布柱状图
data = pd.read_sql(con=engine, sql=sql)
plt.bar(data['户型'], data['total'], label='房源数量')
for x, y in zip(data['户型'], data['total']):
    plt.text(x, y, y, ha='center', va='bottom', fontsize=11)
plt.legend()
plt.show()

In [11]:
# ======================
# 电梯房型分析
# ======================
# 分类统计电梯房与非电梯房
sql = """
select case when 总楼层 > 7 then '电梯房' else '非电梯房' end as tp, count(1) as total, sum(价格)/sum(面积) as per
from `renting-apartment-beijing`
group by case when 总楼层 > 7 then '电梯房' else '非电梯房' end
order by total desc
"""
# 饼图展示占比
data = pd.read_sql(con=engine, sql=sql)
plt.pie(data['total'],
        labels=data['tp'],
        colors=['m','g'],
        startangle=90,
        shadow= True,
        explode=(0,0.1),  # 突出显示非电梯房
        autopct='%1.1f%%')
plt.title('房源数量占比')
plt.show()

# 柱状图展示租金对比
plt.bar(data['tp'], data['per'], label='租金(元/平米)')
for x, y in zip(data['tp'], data['per']):
    plt.text(x, y, y, ha='center', va='bottom', fontsize=11)
plt.legend()
plt.show()

In [12]:
# ======================
# 楼层类型分析
# ======================
# 综合电梯类型与楼层高度的分析
sql = """
select case when 总楼层 > 7 then '电梯房'
            else '非电梯房' end as tp1,
       case when 1.0 * 所在楼层/总楼层 > 0.66 then '高层'
            when 1.0 * 所在楼层/总楼层 > 0.33 then '中层'
            else '低层' end as tp2,
       count(1) as total, sum(价格)/sum(面积) as per
from `renting-apartment-beijing`
group by case when 总楼层 > 7 then '电梯房'
              else '非电梯房' end,
         case when 1.0 * 所在楼层/总楼层 > 0.66 then '高层'
              when 1.0 * 所在楼层/总楼层 > 0.33 then '中层'
              else '低层' end
order by 1, 2  desc
"""
# 折线图展示复合分类结果
data = pd.read_sql(con=engine, sql=sql)
data['floor'] = data['tp1'] + '(' + data['tp2'] + ')'
plt.plot(data['floor'], data['total'], label='房源数量')
plt.plot(data['floor'], data['per'], label='租金(元/平米)')
# 双折线标注数值
for x, y in zip(data['floor'], data['total']):
    plt.text(x, y, y, ha='center', va='bottom', fontsize=11)
for x, y in zip(data['floor'], data['per']):
    plt.text(x, y, y, ha='center', va='bottom', fontsize=11)
plt.legend()
plt.show()

In [13]:
# ======================
# 地铁相关分析
# ======================
# 地铁数量与租金关系
sql = """
select 地铁数, count(1) as total, sum(价格)/sum(面积) as per
from `renting-apartment-beijing`
group by 地铁数
order by 1
"""
data = pd.read_sql(con=engine, sql=sql)
data['地铁数'] = data['地铁数'].astype(str)  # 转换为分类数据
plt.plot(data['地铁数'], data['per'], label='租金(元/平米)')
for x, y in zip(data['地铁数'], data['per']):
    plt.text(x, y, y, ha='center', va='bottom', fontsize=11)
plt.legend()
plt.xlabel('地铁数')
plt.show()

# 地铁距离与租金关系（分段统计）
sql = """
select case when 距离地铁距离 between 0 and 500 then '500米以内'
			when 距离地铁距离 between 501 and 1000 then '1公里以内'
			when 距离地铁距离 between 1001 and 1500 then '1.5公里以内'
			else '1.5公里以外' end as ds,
      count(1) as total, sum(价格)/sum(面积) as per
from `renting-apartment-beijing`
group by case when 距离地铁距离 between 0 and 500 then '500米以内'
			  when 距离地铁距离 between 501 and 1000 then '1公里以内'
			  when 距离地铁距离 between 1001 and 1500 then '1.5公里以内'
        else '1.5公里以外' end
order by 1
"""
# 自定义排序逻辑
data = pd.read_sql(con=engine, sql=sql)
map_dt = {
    '1.5公里以外': 4,
    '1.5公里以内': 3,
    '1公里以内': 2,
    '500米以内': 1
}
data['st'] = data['ds'].apply(lambda x: map_dt[x])  # 添加排序字段
data.sort_values(by='st', inplace=True)
plt.plot(data['ds'], data['per'], label='租金(元/平米)')
for x, y in zip(data['ds'], data['per']):
    plt.text(x, y, y, ha='center', va='bottom', fontsize=11)
plt.legend()
plt.xlabel('距离地铁距离')
plt.show()